# RAG V12 — Multi-Document RAG — Stage 1\n\nRun the single code cell below. It contains the complete Stage 1 implementation and validation for the 27-document multi-cloud corpus.

In [ ]:
from collections import Counter
from pathlib import Path
import re

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# RAG V12 - MULTI-DOCUMENT RAG
# JUPYTER NOTEBOOK - STAGE 1
#
# Entire Stage 1 test in one notebook cell:
#   1. Find all 27 cloud PDFs
#   2. Detect provider from folder
#   3. Load every PDF with PyMuPDF
#   4. Clean text
#   5. Add provider/document/page metadata
#   6. Chunk all documents
#   7. Create unique document-safe chunk IDs
#   8. Build document-safe neighbor lookup
#   9. Run all validation tests
#
# No Pinecone writes yet.
# ============================================================


# ============================================================
# CONFIGURATION
# ============================================================

# This works whether the notebook is started from:
#   repo root
# or
#   repo/notebooks
if Path("data/cloud_documents").exists():
    REPO_ROOT = Path(".")
elif Path("../data/cloud_documents").exists():
    REPO_ROOT = Path("..")
else:
    raise FileNotFoundError(
        "Could not find data/cloud_documents. "
        "Open this notebook from the repository root "
        "or from the notebooks folder."
    )

CLOUD_DOCUMENT_ROOT = (
    REPO_ROOT
    / "data"
    / "cloud_documents"
)

EXPECTED_PROVIDERS = [
    "AWS",
    "Azure",
    "GCP",
    "Oracle",
    "IBM",
]

EXPECTED_DOCUMENT_COUNTS = {
    "AWS": 8,
    "Azure": 5,
    "GCP": 5,
    "Oracle": 4,
    "IBM": 5,
}

EXPECTED_TOTAL_PDFS = 27

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
NEIGHBOR_RANGE = 1


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_cloud_text(text):
    text = text.replace(
        "\t",
        " ",
    )

    # Remove lines containing only page numbers.
    text = re.sub(
        r"(?m)^\s*\d{1,4}\s*$",
        "",
        text,
    )

    # Normalize repeated spaces.
    text = re.sub(
        r"[ ]+",
        " ",
        text,
    )

    # Keep paragraph structure but remove excessive blank lines.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


# ============================================================
# DOCUMENT METADATA HELPERS
# ============================================================

def slugify(text):
    value = text.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "-",
        value,
    )

    return value.strip("-")


def title_from_filename(pdf_path):
    title = pdf_path.stem

    title = re.sub(
        r"[_-]+",
        " ",
        title,
    )

    title = re.sub(
        r"\s+",
        " ",
        title,
    )

    return title.strip()


def get_provider_from_path(pdf_path):
    relative_path = pdf_path.relative_to(
        CLOUD_DOCUMENT_ROOT
    )

    if len(relative_path.parts) < 2:
        raise ValueError(
            "Every PDF must be inside a provider folder. "
            f"Problem file: {pdf_path}"
        )

    provider = relative_path.parts[0]

    if provider not in EXPECTED_PROVIDERS:
        raise ValueError(
            f"Unknown provider folder: {provider}"
        )

    return provider


# ============================================================
# STEP 1 - DISCOVER ALL PDFs
# ============================================================

def discover_cloud_pdfs():
    pdf_files = sorted(
        path
        for path in CLOUD_DOCUMENT_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() == ".pdf"
        )
    )

    if not pdf_files:
        raise FileNotFoundError(
            f"No PDFs found under: {CLOUD_DOCUMENT_ROOT}"
        )

    records = []

    for pdf_path in pdf_files:
        provider = get_provider_from_path(
            pdf_path
        )

        document_id = (
            f"{provider.lower()}__"
            f"{slugify(pdf_path.stem)}"
        )

        records.append(
            {
                "path": pdf_path,
                "provider": provider,
                "document_id": document_id,
                "document_title": title_from_filename(
                    pdf_path
                ),
                "source": pdf_path.name,
            }
        )

    return records


pdf_records = discover_cloud_pdfs()

provider_document_counts = Counter(
    record["provider"]
    for record in pdf_records
)

print(
    "=" * 70
)
print(
    "V12 DOCUMENT DISCOVERY"
)
print(
    "=" * 70
)
print(
    "Total PDFs discovered:",
    len(pdf_records),
)

for provider in EXPECTED_PROVIDERS:
    print(
        f"{provider}:",
        provider_document_counts[
            provider
        ],
    )


# ============================================================
# STEP 2 - LOAD ALL PDF PAGES
# ============================================================

def load_cloud_documents(
    pdf_records,
):
    all_pages = []

    provider_page_counts = Counter()

    for document_number, record in enumerate(
        pdf_records,
        start=1,
    ):
        print(
            f"[{document_number}/{len(pdf_records)}] "
            f"{record['provider']} - "
            f"{record['source']}"
        )

        loader = PyMuPDFLoader(
            str(record["path"])
        )

        pages = loader.load()

        for page in pages:
            page.page_content = (
                clean_cloud_text(
                    page.page_content
                )
            )

            pdf_page = (
                int(
                    page.metadata.get(
                        "page",
                        0,
                    )
                )
                + 1
            )

            page.metadata[
                "pdf_page"
            ] = pdf_page

            page.metadata[
                "provider"
            ] = record[
                "provider"
            ]

            page.metadata[
                "document_id"
            ] = record[
                "document_id"
            ]

            page.metadata[
                "document_title"
            ] = record[
                "document_title"
            ]

            page.metadata[
                "source"
            ] = record[
                "source"
            ]

            page.metadata[
                "source_path"
            ] = str(
                record[
                    "path"
                ]
            )

            provider_page_counts[
                record[
                    "provider"
                ]
            ] += 1

        all_pages.extend(
            pages
        )

    return (
        all_pages,
        provider_page_counts,
    )


print()
print(
    "=" * 70
)
print(
    "LOADING 27 CLOUD DOCUMENTS"
)
print(
    "=" * 70
)

pages, provider_page_counts = (
    load_cloud_documents(
        pdf_records
    )
)

print()
print(
    "Total pages loaded:",
    len(pages),
)

for provider in EXPECTED_PROVIDERS:
    print(
        f"{provider} pages:",
        provider_page_counts[
            provider
        ],
    )


# ============================================================
# STEP 3 - CHUNK ALL DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        "",
    ],
)

chunks = splitter.split_documents(
    pages
)

# Remove completely empty chunks.
chunks = [
    chunk
    for chunk in chunks
    if chunk.page_content.strip()
]

per_document_chunk_counter = Counter()

for global_chunk_number, chunk in enumerate(
    chunks
):
    document_id = chunk.metadata[
        "document_id"
    ]

    document_chunk_number = (
        per_document_chunk_counter[
            document_id
        ]
    )

    chunk.metadata[
        "document_chunk_number"
    ] = document_chunk_number

    chunk.metadata[
        "global_chunk_number"
    ] = global_chunk_number

    chunk.metadata[
        "chunk_characters"
    ] = len(
        chunk.page_content
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"{document_id}__chunk_"
        f"{document_chunk_number:05d}"
    )

    per_document_chunk_counter[
        document_id
    ] += 1


print()
print(
    "=" * 70
)
print(
    "CHUNKING RESULT"
)
print(
    "=" * 70
)
print(
    "Chunk size:",
    CHUNK_SIZE,
)
print(
    "Chunk overlap:",
    CHUNK_OVERLAP,
)
print(
    "Total chunks:",
    len(chunks),
)
print(
    "Documents represented:",
    len(
        per_document_chunk_counter
    ),
)


# ============================================================
# STEP 4 - BUILD DOCUMENT-SAFE CHUNK LOOKUP
# ============================================================

rag_chunk_lookup = {}

for chunk in chunks:
    key = (
        chunk.metadata[
            "document_id"
        ],
        chunk.metadata[
            "document_chunk_number"
        ],
    )

    rag_chunk_lookup[
        key
    ] = chunk


# ============================================================
# STEP 5 - DOCUMENT-SAFE NEIGHBOR FUNCTION
# ============================================================

def get_document_neighbors(
    chunk,
    rag_chunk_lookup,
    neighbor_range=1,
):
    document_id = chunk.metadata[
        "document_id"
    ]

    current_chunk_number = (
        chunk.metadata[
            "document_chunk_number"
        ]
    )

    neighbors = []

    for offset in range(
        -neighbor_range,
        neighbor_range + 1,
    ):
        key = (
            document_id,
            current_chunk_number
            + offset,
        )

        neighbor = (
            rag_chunk_lookup.get(
                key
            )
        )

        if neighbor is not None:
            neighbors.append(
                neighbor
            )

    return neighbors


# ============================================================
# STEP 6 - SHOW ONE SAMPLE CHUNK
# ============================================================

sample_chunk = chunks[
    len(chunks) // 2
]

print()
print(
    "=" * 70
)
print(
    "SAMPLE V12 CHUNK"
)
print(
    "=" * 70
)

print(
    "Provider:",
    sample_chunk.metadata[
        "provider"
    ],
)

print(
    "Document:",
    sample_chunk.metadata[
        "document_title"
    ],
)

print(
    "Document ID:",
    sample_chunk.metadata[
        "document_id"
    ],
)

print(
    "PDF page:",
    sample_chunk.metadata[
        "pdf_page"
    ],
)

print(
    "Document chunk number:",
    sample_chunk.metadata[
        "document_chunk_number"
    ],
)

print(
    "Chunk ID:",
    sample_chunk.metadata[
        "chunk_id"
    ],
)

print()
print(
    sample_chunk.page_content[
        :1000
    ]
)


# ============================================================
# STEP 7 - TEST DOCUMENT-SAFE NEIGHBORS
# ============================================================

sample_neighbors = (
    get_document_neighbors(
        sample_chunk,
        rag_chunk_lookup,
        neighbor_range=
            NEIGHBOR_RANGE,
    )
)

print()
print(
    "=" * 70
)
print(
    "SAMPLE NEIGHBORS"
)
print(
    "=" * 70
)

for neighbor in sample_neighbors:
    print(
        neighbor.metadata[
            "provider"
        ],
        "|",
        neighbor.metadata[
            "document_id"
        ],
        "| chunk",
        neighbor.metadata[
            "document_chunk_number"
        ],
        "| page",
        neighbor.metadata[
            "pdf_page"
        ],
    )


# ============================================================
# STEP 8 - VALIDATION TESTS
# ============================================================

test_results = []


def check(
    condition,
    label,
):
    if condition:
        print(
            f"{label}: PASS ✅"
        )

        test_results.append(
            True
        )

    else:
        print(
            f"{label}: FAIL ❌"
        )

        test_results.append(
            False
        )


print()
print(
    "=" * 70
)
print(
    "RAG V12 STAGE 1 VALIDATION"
)
print(
    "=" * 70
)


# TEST 1
check(
    len(pdf_records)
    == EXPECTED_TOTAL_PDFS,
    "27 PDFs discovered",
)


# TEST 2
check(
    set(
        provider_document_counts
    )
    ==
    set(
        EXPECTED_PROVIDERS
    ),
    "All 5 cloud providers discovered",
)


# TESTS 3-7
for provider in EXPECTED_PROVIDERS:
    check(
        provider_document_counts[
            provider
        ]
        ==
        EXPECTED_DOCUMENT_COUNTS[
            provider
        ],
        (
            f"{provider} document count "
            f"= "
            f"{EXPECTED_DOCUMENT_COUNTS[provider]}"
        ),
    )


# TEST 8
document_ids = [
    record[
        "document_id"
    ]
    for record in pdf_records
]

check(
    len(document_ids)
    ==
    len(
        set(
            document_ids
        )
    ),
    "Document IDs are unique",
)


# TEST 9
check(
    len(pages) > 0,
    "PDF pages loaded",
)


# TEST 10
required_page_metadata = {
    "pdf_page",
    "provider",
    "document_id",
    "document_title",
    "source",
    "source_path",
}

check(
    all(
        required_page_metadata.issubset(
            page.metadata
        )
        for page in pages
    ),
    "Every page has V12 document metadata",
)


# TEST 11
page_document_ids = {
    page.metadata[
        "document_id"
    ]
    for page in pages
}

check(
    len(
        page_document_ids
    )
    ==
    EXPECTED_TOTAL_PDFS,
    "All 27 documents produced pages",
)


# TEST 12
check(
    len(chunks)
    >
    len(pages),
    "Documents split into chunks",
)


# TEST 13
required_chunk_metadata = (
    required_page_metadata
    |
    {
        "document_chunk_number",
        "global_chunk_number",
        "chunk_characters",
        "chunk_id",
    }
)

check(
    all(
        required_chunk_metadata.issubset(
            chunk.metadata
        )
        for chunk in chunks
    ),
    "Every chunk has V12 chunk metadata",
)


# TEST 14
chunk_ids = [
    chunk.metadata[
        "chunk_id"
    ]
    for chunk in chunks
]

check(
    len(chunk_ids)
    ==
    len(
        set(
            chunk_ids
        )
    ),
    "Chunk IDs are unique",
)


# TEST 15
chunk_document_ids = {
    chunk.metadata[
        "document_id"
    ]
    for chunk in chunks
}

check(
    len(
        chunk_document_ids
    )
    ==
    EXPECTED_TOTAL_PDFS,
    "All 27 documents produced chunks",
)


# TEST 16
check(
    len(
        rag_chunk_lookup
    )
    ==
    len(chunks),
    "Document chunk lookup is complete",
)


# TEST 17
check(
    all(
        neighbor.metadata[
            "document_id"
        ]
        ==
        sample_chunk.metadata[
            "document_id"
        ]
        for neighbor in sample_neighbors
    ),
    "Neighbor expansion stays inside one PDF",
)


# ============================================================
# FINAL RESULT
# ============================================================

tests_passed = sum(
    test_results
)

total_tests = len(
    test_results
)

accuracy = (
    tests_passed
    /
    total_tests
    *
    100
)


print()
print(
    "=" * 70
)
print(
    "RAG V12 MULTI-DOCUMENT INGESTION TEST"
)
print(
    "=" * 70
)
print(
    "Documents:",
    len(pdf_records),
)
print(
    "Pages:",
    len(pages),
)
print(
    "Chunks:",
    len(chunks),
)
print(
    f"Tests passed: "
    f"{tests_passed}/{total_tests}"
)
print(
    f"Accuracy: "
    f"{accuracy:.1f}%"
)


if tests_passed == total_tests:
    print(
        "V12 STAGE 1 MULTI-DOCUMENT INGESTION: PASS ✅"
    )

else:
    print(
        "V12 STAGE 1 MULTI-DOCUMENT INGESTION: FAIL ❌"
    )


# ============================================================
# OPTIONAL: PROVIDER SUMMARY
# ============================================================

provider_chunk_counts = Counter(
    chunk.metadata[
        "provider"
    ]
    for chunk in chunks
)

print()
print(
    "=" * 70
)
print(
    "PROVIDER SUMMARY"
)
print(
    "=" * 70
)

for provider in EXPECTED_PROVIDERS:
    print(
        f"{provider:<8}",
        "| PDFs:",
        provider_document_counts[
            provider
        ],
        "| Pages:",
        provider_page_counts[
            provider
        ],
        "| Chunks:",
        provider_chunk_counts[
            provider
        ],
    )
